# 2 · Train MedCAT with a MedRoBERTa.nl Status classifier

Input: the model pack from notebook 1 and one or more **MedCATtrainer exports** (download *with text*).

1. **Supervised NER+L training:** teaches MedCAT the validated symptom spans and their SNOMED CT concepts.
2. **MetaCAT `Status` (Affirmed / Negated):** trains a classifier on **`CLTL/MedRoBERTa.nl`** representations of each mention in its context.
3. Saves:
   - `..._medroberta_<version>.zip`: the full model (NER+L + MetaCAT) for evaluation (notebook 3) and deployment,
   - `..._annotation_<version>.zip`: the same model without MetaCAT, for the next annotation round in MedCATtrainer.

Each run trains the MetaCAT from scratch. To add a new annotation round, list **all** train exports so far in `TRAIN_EXPORTS`. Never include test exports here.

## Configuration

In [ ]:
from pathlib import Path

BASE_MODEL_PACK = "medcat_data/models/snomed_symptoms_en_nl_annotation_round1.zip"  # from notebook 1 (no MetaCAT)
TRAIN_EXPORTS = [
    "medcat_data/exports/round1_train_export.json",
]

MODEL_NAME = "snomed_symptoms_en_nl"
VERSION_TAG = "v1"                 # names the output model packs
OVERWRITE_EXISTING = False         # True: replace existing model packs with the same VERSION_TAG

# Supervised NER+L
NER_EPOCHS = 3

# MetaCAT Status classifier
STATUS_TASK = "Status"
STATUS_VALUES = ["Affirmed", "Negated"]   # must match the MedCATtrainer meta task values
MEDROBERTA_MODEL = "CLTL/MedRoBERTa.nl"
METACAT_EPOCHS = 10
METACAT_BATCH_SIZE = 8
METACAT_VAL_FRACTION = 0.1         # held out from the train export to pick the best epoch
# MedRoBERTa.nl encodes the mention in context; MetaCAT trains a classifier on top of it.
# Tested (MedCAT 2.10, synthetic NL/EN data): frozen encoder + lr 1e-3 reached test F1 ~0.9.
# Fine-tuning the whole encoder (False, lr 2e-5..1e-4) did not learn at all: one class for everything.
FREEZE_ENCODER = True
METACAT_LR = 1e-3                  # for FREEZE_ENCODER = False use ~2e-5 and check the validation report
CONTEXT_TOKENS = 15                # tokens of context on each side of the mention

DATA_DIR = Path("medcat_data")
MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def free_pack_name(name):
    """Model pack folder + zip for `name`; MedCAT won't overwrite an existing pack."""
    folder = MODELS_DIR / name
    zip_path = folder.with_suffix(".zip")
    if folder.exists() or zip_path.exists():
        if not OVERWRITE_EXISTING:
            raise FileExistsError(f"{zip_path} already exists. Change the tag in the configuration, "
                                  "or set OVERWRITE_EXISTING = True to replace it.")
        import shutil
        shutil.rmtree(folder, ignore_errors=True)
        zip_path.unlink(missing_ok=True)
    return name

## MedRoBERTa.nl patch
MedCAT's MetaCAT only supports BERT models (`BertModel` / `BertTokenizerFast`). MedRoBERTa.nl is a RoBERTa model, so both loaders are redirected to Hugging Face's `Auto*` classes. Apply this before creating or loading a MetaCAT.

In [ ]:
from transformers import AutoModel, AutoTokenizer
import medcat.components.addons.meta_cat.models as _mc_models
import medcat.components.addons.meta_cat.mctokenizers.bert_tokenizer as _mc_bert_tok

_mc_models.BertModel = AutoModel
_mc_bert_tok.BertTokenizerFast = AutoTokenizer

## Load and check the annotations

In [ ]:
import json
from collections import Counter
import pandas as pd


def meta_anns_of(ann):
    """MedCATtrainer exports meta_anns as a list (or, in older exports, a dict keyed by task)."""
    m = ann.get("meta_anns") or []
    return list(m.values()) if isinstance(m, dict) else list(m)


def is_gold(ann):
    """A human-validated, correct (or corrected) concept annotation."""
    return (ann.get("validated", True) and (ann.get("correct", False) or ann.get("alternative", False))
            and not ann.get("deleted", False) and not ann.get("killed", False) and not ann.get("irrelevant", False))


train_data = {"projects": []}
for path in TRAIN_EXPORTS:
    with open(path, encoding="utf-8") as f:
        train_data["projects"].extend(json.load(f)["projects"])

docs = [d for p in train_data["projects"] for d in p["documents"]]
anns = [a for d in docs for a in d.get("annotations", [])]
gold = [a for a in anns if is_gold(a)]
status = Counter(m["value"] for a in gold for m in meta_anns_of(a) if m.get("name") == STATUS_TASK)
print(f"{len(train_data['projects'])} project(s), {len(docs)} documents, {len(anns)} annotations, {len(gold)} correct")
print("Status labels:", dict(status))
print("\nMost frequent concepts:")
print(pd.Series([a["cui"] for a in gold]).value_counts().head(15).to_string())

unknown = set(status) - set(STATUS_VALUES)
if unknown:
    raise ValueError(f"Status values {unknown} are not in STATUS_VALUES={STATUS_VALUES}")
if len(status) < 2:
    print(f"\nWARNING: Status labels for fewer than 2 classes. MetaCAT can't learn to tell them apart.")

## Supervised NER+L training

In [ ]:
from medcat.cat import CAT

cat = CAT.load_model_pack(BASE_MODEL_PACK)
if cat.get_addons():
    print("Note: the base pack already contains addons:", [a.name for a in cat.get_addons()])

cat.trainer.train_supervised_raw(train_data, nepochs=NER_EPOCHS, use_filters=False, test_size=0,
                                 train_addons=False, disable_progress=True)
print("NER+L training done.")

## Train the MetaCAT Status classifier on MedRoBERTa.nl

In [ ]:
from medcat.config.config_meta_cat import ConfigMetaCAT
from medcat.components.addons.meta_cat.meta_cat import MetaCATAddon

mc_config = ConfigMetaCAT()
mc_config.general.category_name = STATUS_TASK
mc_config.general.category_value2id = {v: i for i, v in enumerate(STATUS_VALUES)}
mc_config.general.tokenizer_name = "bert-tokenizer"  # Hugging Face tokenizer wrapper (AutoTokenizer via the patch)
mc_config.general.lowercase = False                    # MedRoBERTa.nl's tokenizer is cased
mc_config.general.cntx_left = CONTEXT_TOKENS
mc_config.general.cntx_right = CONTEXT_TOKENS

mc_config.model.model_name = "bert"                    # transformer backbone (AutoModel via the patch)
mc_config.model.model_variant = MEDROBERTA_MODEL
mc_config.model.nclasses = len(STATUS_VALUES)
mc_config.model.input_size = 768                       # MedRoBERTa.nl hidden size
mc_config.model.num_layers = 12                        # all encoder layers (MedCAT's default is 2)
mc_config.model.model_freeze_layers = FREEZE_ENCODER

mc_config.train.nepochs = METACAT_EPOCHS
mc_config.train.lr = METACAT_LR
mc_config.train.batch_size = METACAT_BATCH_SIZE
mc_config.train.test_size = METACAT_VAL_FRACTION
mc_config.train.compute_class_weights = True           # Negated is usually rarer than Affirmed

meta_cat_addon = MetaCATAddon.create_new(mc_config, cat.pipe.tokenizer)
mc = meta_cat_addon.mc
metacat_dir = MODELS_DIR / f"metacat_{STATUS_TASK.lower()}_{VERSION_TAG}"
metacat_dir.mkdir(parents=True, exist_ok=True)
report = mc.train_from_json(TRAIN_EXPORTS, save_dir_path=str(metacat_dir), overwrite=True)
print("Validation report (best epoch):")
print(pd.DataFrame(report.get("report", report)).T.round(3).to_string())

cat.add_addon(meta_cat_addon)

## Save the model packs

In [ ]:
import copy

full_pack = cat.save_model_pack(str(MODELS_DIR), pack_name=free_pack_name(f"{MODEL_NAME}_medroberta_{VERSION_TAG}"),
                                add_hash_to_pack_name=False) + ".zip"

# Same CDB + Vocab without MetaCAT: MedCATtrainer uses stock MedCAT, which can't load the MedRoBERTa.nl MetaCAT
cnf = copy.deepcopy(cat.config)
cnf.components.addons = []
annotation_pack = CAT(cdb=copy.deepcopy(cat.cdb), vocab=cat.vocab, config=cnf).save_model_pack(
    str(MODELS_DIR), pack_name=free_pack_name(f"{MODEL_NAME}_annotation_{VERSION_TAG}"), add_hash_to_pack_name=False) + ".zip"

print("Full model (evaluate / deploy):     ", full_pack)
print("Next annotation round (MedCATtrainer):", annotation_pack)

## Quick check

In [ ]:
for text in ["Patiënt heeft last van koorts en benauwdheid, maar geen pijn op de borst.",
             "Patient denies shortness of breath, reports mild fatigue and cough."]:
    print("\n" + text)
    for e in cat.get_entities(text)["entities"].values():
        s = e["meta_anns"][STATUS_TASK]
        print(f"  -> {e['source_value']!r}  {e['cui']} ({e['pretty_name']})  {s['value']} ({s['confidence']:.2f})")

Next: evaluate `..._medroberta_<version>.zip` on the test export with `03_medcat_evaluate.ipynb`.